# Pasquereau & Turner (2013) on DANDI:001636

Paper: Pasquereau B, Turner RS (2013). Primary motor cortex of the parkinsonian monkey: altered
neuronal responses to muscle stretch. *Front. Syst. Neurosci.* 7:98,
[doi:10.3389/fnsys.2013.00098](https://doi.org/10.3389/fnsys.2013.00098).

Dataset: [DANDI:001636](https://dandiarchive.org/dandiset/001636/0.261124.1906), version
`0.261124.1906` (doi:10.48324/dandi.001636/0.261124.1906). This version was asked for in the
prompt of this attempt; it is also the newest published version (published 2026-11-24). The
paper cites no version.

Claims: `claims/pasquereau-2013.yaml`, written from the full text in PubMed Central
(PMC3840326), its Methods, Results, Table 1, figure legends and supplementary data sheet
(DataSheet1.PDF), whose table gives the results for each monkey.

The files of monkey L mark no torque perturbation, so every claim is tested on monkey V
(Venus, elbow), as the claims file says.

| Claim | Tested or blocked |
|---|---|
| cell-counts | tested |
| monkey-l-perturbations | blocked, data missing |
| baseline-rate | tested |
| ptn-faster-than-csn | tested |
| antidromic-latency | tested |
| false-positive-rate | computed; blocked, methods underspecified (the false-positive rate is far from the paper's) |
| torque-related-fractions, prevalence-unchanged, ptn-more-responsive, one-direction-half, excitatory-majority, magnitude-unchanged, venus-response-magnitudes, latency-shorter, latency-not-training, fwhm-shorter-ptn, dsi-reduced, venus-dsi, dsi-not-baseline, velocity-matched-magnitude, session-correlations | blocked through false-positive-rate; their results are computed and shown, not deciding |
| kinematics-slower | tested |
| kinematic-latency-unchanged | tested |
| emg-m2-larger | tested |
| emg-duration-longer | tested |
| emg-onset-unchanged | tested |
| emg-m1-weak | tested |
| trial-correlations | blocked, methods underspecified |

Method details come from the claims file, which cites the Methods; details it lacks are taken
from the paper and cited in `pasq.py`'s docstring and below.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import pynwb
import remfile
import h5py
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

import pasq

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
t0 = time.time()
venus_urls = pasq.asset_urls("sub-Venus")
leu_urls = pasq.asset_urls("sub-Leu")
print(f"{len(venus_urls)} Venus files, {len(leu_urls)} Leu files at version {pasq.DANDISET_VERSION}")

## What one file holds

One Venus file opened with pynwb: its acquisition and processing series, its interval tables
with every column, and the units table's columns.

In [ ]:
path, url = venus_urls[0]
with pynwb.NWBHDF5IO(file=h5py.File(remfile.File(url), "r"), load_namespaces=True) as io:
    nwb = io.read()
    print(path)
    for name, ts in nwb.acquisition.items():
        print(f"acquisition {name}: {ts.data.shape}, unit {ts.unit}")
    for mod_name, mod in nwb.processing.items():
        for name, obj in mod.data_interfaces.items():
            print(f"processing/{mod_name}/{name}: {type(obj).__name__}")
    for name, table in nwb.intervals.items():
        print(f"intervals {name}: {len(table)} rows, columns {list(table.colnames)}")
    print("units columns", list(nwb.units.colnames))
    print(nwb.units.to_dataframe().drop(columns=["spike_times", "obs_intervals", "electrodes"]).T)

## Loading

Every Venus file is streamed once; only windows around each perturbation are kept (spikes from
-1500 to +500 ms, angle -200 to +400 ms, EMG -300 to +300 ms, torque command -50 to +100 ms).
The Leu files are read for their trials' perturbation columns, torque command and units table.

In [ ]:
venus = pasq.load_all(venus_urls)
leu = pasq.load_meta_all(leu_urls)
print(f"loaded in {time.time() - t0:.0f} s")

In [ ]:
cells = pasq.cell_responses(venus)
cells["unit"] = cells.groupby("file").cumcount()
cells["studied"] = (cells.n_fle >= pasq.MIN_PERTURBATIONS) & (cells.n_ext >= pasq.MIN_PERTURBATIONS)
studied = cells[cells.studied].copy()
studied["state"] = np.where(studied.post, "post", "pre")
PTN, CSN = "pyramidal_tract_neuron", "corticostriatal_neuron"
groups = {"M1": studied.index == studied.index, "PTN": (studied.ptype == PTN).values,
          "CSN": (studied.ptype == CSN).values}
first_day = pd.to_datetime(cells.date).min()

## Data check

In [ ]:
all_units = pd.concat([x["units"].assign(subject="Venus", path=x["path"]) for x in venus]
                      + [x["units"].assign(subject="Leu", path=x["path"]) for x in leu], ignore_index=True)
print("units by subject, MPTP state and projection type")
print(all_units.groupby(["subject", "post", "ptype"]).size().to_string())
print("unit names repeated across files:", all_units.unit_name.duplicated().sum())
fname_post = all_units.path.str.contains("PostMPTP")
print("units whose is_post_mptp disagrees with the file name:", int((fname_post != all_units.post).sum()))

pert = pd.concat([x["pert"].assign(file=i, post=bool(x["units"].post.iloc[0])) for i, x in enumerate(venus)])
print("\nVenus perturbations by state and direction")
print(pert.groupby(["post", "torque"]).size().to_string())
print("Venus files without any perturbation:", sum(len(x["pert"]) == 0 for x in venus))
print("Leu trials by torque_perturbation_type:",
      pd.Series(np.concatenate([x["torque_type"] for x in leu])).value_counts().to_dict())
print("Leu perturbation onset times that are not NaN:", int(sum(np.isfinite(x["onset"]).sum() for x in leu)))

dates = pd.to_datetime(cells.date)
print("\nVenus recording dates: pre", dates[~cells.post].min().date(), "to", dates[~cells.post].max().date(),
      "; post", dates[cells.post].min().date(), "to", dates[cells.post].max().date())
delay = (pert.onset - pert.center)
print("perturbation onset after centre target appearance (s): median %.3f, 5th to 95th percentile %.3f to %.3f"
      % (delay.median(), delay.quantile(0.05), delay.quantile(0.95)))
print("units with fewer observed perturbations than their file has:",
      int((cells.n_fle + cells.n_ext < [len(venus[f]["pert"]) for f in cells.file]).sum()))

ang = np.concatenate([x["angle"].ravel() for x in venus if x["angle"] is not None and len(x["pert"])])
print("\nelbow angle around perturbations (degrees): %.1f to %.1f" % (np.nanmin(ang), np.nanmax(ang)))
emg_vals = {ch: np.concatenate([x["emg"][ch].ravel() for x in venus if ch in x["emg"]]) for ch in pasq.EMG_CHANNELS}
for ch, v in emg_vals.items():
    print(f"{ch}: {sum(ch in x['emg'] for x in venus)} files, values {np.nanmin(v):.0f} to {np.nanmax(v):.0f} (a.u.)")
print("SDF kernel: gamma k = %.1f, theta = %.0f ms, mode %.0f ms, mean %.0f ms; t threshold %.3f"
      % (pasq.GAMMA_K, pasq.GAMMA_THETA_MS, (pasq.GAMMA_K - 1) * pasq.GAMMA_THETA_MS,
         pasq.GAMMA_K * pasq.GAMMA_THETA_MS, pasq.T_CRIT))

### Recordings deposited twice

Each trial's spikes, relative to its trial start, and each perturbation's angle and EMG windows
are compared across all Venus files.

In [ ]:
from collections import defaultdict
from itertools import combinations

def shared(keys_by_file):
    owners = defaultdict(set)
    for f, keys in keys_by_file.items():
        for k in keys:
            owners[k].add(f)
    pairs = defaultdict(int)
    for fs in owners.values():
        for a, b in combinations(sorted(fs), 2):
            pairs[(a, b)] += 1
    return pairs

spike_keys = {}
for fi, x in enumerate(venus):
    keys = []
    for ui, st in enumerate(x["spikes_all"]):
        for a, b in zip(x["trials"].start, x["trials"].stop):
            s = np.round(st[(st >= a) & (st < b)] - a, 4)
            if len(s) >= 5:
                keys.append(tuple(s))
    spike_keys[fi] = keys
analog_keys = {}
for fi, x in enumerate(venus):
    keys = []
    for name, arr in [("angle", x["angle"])] + list(x["emg"].items()):
        if arr is None:
            continue
        keys += [(name,) + tuple(np.round(w[::7], 2)) for w in arr if not np.isnan(w).any()]
    analog_keys[fi] = keys
print("pairs of units sharing a trial's spike train:", len(shared(spike_keys)))
print("pairs of files sharing an angle or EMG window:", len(shared(analog_keys)))

### Clocks

The torque command, the elbow angle and the spikes are checked against the trials table's
perturbation onsets. The command should leave rest a few ms after onset and the joint should move
in the torque's direction after it. For the spikes, a contrast: threshold crossings by the claims
file's detection after the real onset against a pseudo onset 600 ms earlier and against trials
without a perturbation aligned where the file's perturbations fall.

In [ ]:
zt = -pasq.TORQUE_WINDOW_MS[0]
for post in [False, True]:
    cmd = []
    for x in venus:
        if x["torque_cmd"] is None or bool(x["units"].post.iloc[0]) != post or len(x["pert"]) == 0:
            continue
        s = np.where(x["pert"].torque.values == "flexion", 1.0, -1.0)[:, None]
        cmd.append(s * (x["torque_cmd"] - x["torque_cmd"][:, :zt].mean(1, keepdims=True)))
    m = np.nanmean(np.vstack(cmd), 0)
    half = np.where(m[zt:] > m[zt:zt + 50].max() / 2)[0][0]
    leave = np.where(m[zt:] > 0.05 * m[zt:zt + 50].max())[0][0]
    print(f"{'post' if post else 'pre'}-MPTP: mean command leaves rest (5% of pulse) at {leave} ms and reaches half its peak at {half} ms after onset")
za = -pasq.ANGLE_WINDOW_MS[0]
for dname, sgn in [("flexion", 1), ("extension", -1)]:
    a = np.vstack([x["angle"][x["pert"].torque.values == dname] for x in venus if x["angle"] is not None and len(x["pert"])])
    disp = np.nanmean(a - a[:, [za]], 0)
    print(f"{dname} torque: mean angle change at 0, 20, 50, 100 ms: "
          + ", ".join(f"{disp[za + t]:+.2f}" for t in [0, 20, 50, 100]) + " deg (positive = flexion)")

pr = pasq.pseudo_readings(venus, set(zip(studied.file, studied.unit)))
nt = pasq.no_torque_crossings(venus, set(zip(studied.file, studied.unit)))
print("\nany crossing of the claims file's threshold over 0 to 200 ms, per cell-direction:")
print("  after torque onset %.3f (n = %d), after pseudo onset %.3f, after the no-torque alignment %.3f (n = %d cells)"
      % (pr[pr.onset == "torque"].mean_sd.mean(), (pr.onset == "torque").sum(),
         pr[pr.onset == "pseudo"].mean_sd.mean(), nt.any_crossing.mean(), len(nt)))
on = np.concatenate([studied.fle_onset.dropna(), studied.ext_onset.dropna()])
print("response onsets after torque (ms): median %.0f, inside 20 to 60 by construction" % np.median(on))

The command leaves rest a few ms after the marked onset and the joint moves in the torque's
direction, so the analog series and the trials table share a clock (consistent with the data, as
far as a mean trace can show it). Spikes cross the threshold far more often after the real onset
than after the pseudo onset or in trials without a torque, so the spike times share it too.

### Findings about the data

Each cell below shows one problem, loading only what it needs.

In [ ]:
# Monkey L: no perturbation times, and no pulses in the torque command.
types = pd.Series(np.concatenate([x["torque_type"] for x in leu])).value_counts().to_dict()
print("Leu trials by torque_perturbation_type:", types)
finite = [x for x in leu if x["torque_cmd_finite"] > 0]
print("Leu files whose torque command has any finite value:", len(finite), "of", len(leu),
      "; of these after MPTP:", sum(bool(x["units"].post.iloc[0]) for x in finite))

In [ ]:
# The torque command's pulse is about three times larger after MPTP; the paper gives one 0.1 Nm pulse.
kin_trials, kin_sessions = pasq.session_kinematics(venus)
print(kin_trials.groupby("post").command.describe().round(1))
print("\nsessions by median pulse size (a.u.), pre and post:")
print(pd.crosstab(kin_trials.groupby("file").post.first(),
                  pd.cut(kin_trials.groupby("file").command.median(), [0, 30, 60, 120, 250, 400])))

In [ ]:
# Torque-evoked displacements: the paper's mean is 10 degrees, and slower, smaller displacements after MPTP.
print(kin_sessions.groupby(["post", "direction"])[["peak_velocity", "amplitude", "latency"]].mean().round(2))
print("perturbations with displacement >= 10 deg: %.1f%%" % (100 * (kin_trials.amplitude >= 10).mean()))
print("sessions with mean peak velocity 130 to 200 deg/s:",
      int(kin_sessions.peak_velocity.between(130, 200).sum()), "of", len(kin_sessions))

## Claims

### cell-counts

> In monkey V, 112 neurons were studied before MPTP and 159 after, of which 49 and 54 were PTNs
> and 31 and 48 CSNs; three neurons in all were activated from both the peduncle and the putamen.

Criterion: counts of monkey V units with at least 5 perturbations per direction, by MPTP state and
projection type; dual-projecting units counted over all files.

In [ ]:
expected = {"M1": (112, 159), "PTN": (49, 54), "CSN": (31, 48)}
for g, m in groups.items():
    n = studied[m].groupby("post").size()
    print(f"{g}: pre {n.get(False, 0)} (expected {expected[g][0]}), post {n.get(True, 0)} (expected {expected[g][1]})")
dual = all_units[all_units.ptype == "pyramidal_tract_and_corticostriatal_neuron"]
print("activated from both peduncle and putamen, all files:", len(dual), dual.groupby("subject").size().to_dict(),
      "; of these studied:", int((studied.ptype == "pyramidal_tract_and_corticostriatal_neuron").sum()))
print("other types among the studied cells:", studied[~studied.ptype.isin([PTN, CSN])].groupby(["post", "ptype"]).size().to_dict())
print("Venus units left out for fewer than 5 perturbations in a direction:",
      cells[~cells.studied].groupby("post").size().to_dict())

**Result: reproduced.** 115 and 157 cells before and after MPTP (112 and 159 expected), 51 and 54
PTNs (49 and 54), 32 and 48 CSNs (31 and 48), and 3 dual-projecting cells, all within tolerance.

### monkey-l-perturbations

> In monkey L, 57 of 115 neurons before MPTP and 35 of 73 after responded to torque perturbations
> of the wrist, and the monkey L responses enter all pooled results.

Blocked: no Leu trial marks a perturbation (cell `finding-leu-no-perturbations`), and the torque
command is NaN in most files.

In [ ]:
print("Leu units by state:", all_units[all_units.subject == "Leu"].groupby("post").size().to_dict())
print("Leu trials with a perturbation onset:", int(sum(np.isfinite(x["onset"]).sum() for x in leu)))

**Result: blocked (data missing).** The Leu files hold 138 and 72 units but no perturbation
times, and the torque command is NaN in 114 of 148 files (the other 34, all after MPTP, hold a trace that
the file description says has no distinct pulses).

### baseline-rate

> Before torque onset, the firing rate of M1 neurons was lower after MPTP, because PTN rates fell
> (Mann-Whitney p < 0.001) while CSN rates did not change (p = 0.5). In monkey V the rates were 12.6
> and 8.9 spikes/s for all cells, 20.3 and 14.8 for PTNs, and 3.5 and 3.4 for CSNs.

Baseline rate: the mean over all perturbations of the SDF over the 500 ms before onset (claims).

In [ ]:
exp = {"M1": (12.6, 1.98, 8.9, 1.33), "PTN": (20.3, 2.46, 14.8, 2.72), "CSN": (3.5, 1.47, 3.4, 1.01)}
for g, m in groups.items():
    s = studied[m]
    a, b = s[~s.post].baseline_rate, s[s.post].baseline_rate
    p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    e = exp[g]
    print(f"{g}: pre {a.mean():.2f} +/- {a.std():.2f} (expected {e[0]} +/- {e[1]}, n {len(a)}), "
          f"post {b.mean():.2f} +/- {b.std():.2f} (expected {e[2]} +/- {e[3]}, n {len(b)}); "
          f"change {100 * (b.mean() / a.mean() - 1):+.0f}%, Mann-Whitney p = {p:.3g}")

**Result: reproduced.** All six means are within tolerance (M1 12.66 and 8.85, PTN 18.49 and
14.37, CSN 3.48 and 3.36 spikes/s); the PTN rate is lower after MPTP (Mann-Whitney p = 0.020) and
the CSN rate does not differ (p = 0.65). The PTN test does not reach the paper's p < 0.001; the M1
rate is also lower (p = 0.0022, below the paper's 0.05/2).

### ptn-faster-than-csn

> Before MPTP, PTNs fired much faster than CSNs at rest (Mann-Whitney p < 0.001).

In [ ]:
pre = studied[~studied.post]
a, b = pre[pre.ptype == PTN].baseline_rate, pre[pre.ptype == CSN].baseline_rate
print(f"pre-MPTP PTN {a.mean():.2f} spikes/s (range {a.min():.1f} to {a.max():.1f}, n {len(a)}), "
      f"CSN {b.mean():.2f} (range {b.min():.1f} to {b.max():.1f}, n {len(b)}); "
      f"Mann-Whitney p = {stats.mannwhitneyu(a, b, alternative='two-sided').pvalue:.2g}")
print("reported, not judged: text 16.3 and 3.6, Table 1 18.3 and 3.2 spikes/s (both monkeys)")

**Result: reproduced.** PTNs 18.49 against CSNs 3.48 spikes/s, Mann-Whitney p far below 0.001.

### antidromic-latency

> PTN antidromic latencies did not change after MPTP (1.9 ms, p = 0.9), while CSN latencies were
> longer (5.1 against 6.1 ms, p < 0.05/3).

In [ ]:
exp = {"PTN": (1.9, 0.22, 1.9, 0.2), "CSN": (5.1, 0.61, 6.1, 0.89)}
leu_units = all_units[all_units.subject == "Leu"]
for label, data in [("monkey V studied", studied), ("variant, + every Leu unit", None)]:
    for g, t in [("PTN", PTN), ("CSN", CSN)]:
        s = studied[studied.ptype == t][["post", "latency"]]
        if data is None:
            s = pd.concat([s, leu_units[leu_units.ptype == t][["post", "latency"]]])
        a, b = s[~s.post].latency.dropna(), s[s.post].latency.dropna()
        p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
        e = exp[g]
        print(f"{label} {g}: pre {a.mean():.2f} ms (expected {e[0]} +/- {e[1]}, n {len(a)}), "
              f"post {b.mean():.2f} (expected {e[2]} +/- {e[3]}, n {len(b)}), Mann-Whitney p = {p:.3g}")

**Result: partially reproduced.** All four means are within tolerance (PTN 1.92 and 1.86 ms, CSN
4.91 and 6.85 ms) and the PTN latency does not differ (p = 0.68), but the CSN latency, longer after
MPTP, has p = 0.019, above the 0.05/3 the criterion asks. With every Leu unit added (variant), the
CSN test gives p = 0.0072.

### false-positive-rate

> The detection threshold, p < 0.01 divided by 5 independent comparisons, holds the chance of a
> false response over the 200 ms epoch near 0.01.

Criterion: the same detection at a pseudo onset 600 ms before the torque, on the studied cells,
both directions; reproduced when the fraction of cell-directions with any threshold crossing over
0 to 200 ms is between 0.0033 and 0.03. The other readings of the Methods' test (step 1.6) are
shown with the fraction of real cell-directions each would call responsive.

In [ ]:
pse = pr[pr.onset == "pseudo"]
print("claims file's detection, pseudo onset: any crossing over 0 to 200 ms %.3f, response with onset in 20 to 60 ms %.3f (n = %d cell-directions)"
      % (pse.mean_sd.mean(), np.r_[studied.pse_fle_responds, studied.pse_ext_responds].mean(), len(pse)))
print("no-torque trials, same detection: any crossing %.3f, response 20 to 60 ms %.3f (n = %d cells)"
      % (nt.any_crossing.mean(), nt.responds.mean(), len(nt)))
print("\nreadings of the test (fraction of cell-directions with any crossing over 0 to 200 ms):")
print(pr.groupby("onset")[["mean_sd", "ttest", "trial_se", "trial_sd"]].mean().round(3).T)
for label, s in [("pre", studied[~studied.post]), ("post", studied[studied.post])]:
    print(f"pseudo any-crossing rate {label}-MPTP: "
          f"{np.r_[s.pse_fle_any_crossing, s.pse_ext_any_crossing].mean():.3f}")

**Result: blocked (methods underspecified).** The claims file's detection crosses its threshold in
0.23 of cell-directions after a pseudo onset (0.20 in trials without a torque), far above the
paper's 0.01 and outside 0.0033 to 0.03, so this procedure is not the paper's. Of the other
readings, only a threshold from the SD of all single-trial baseline values comes near 0.01
(0.013), and it calls only 0.20 of real cell-directions responsive, far below the paper's 62 to
75% of cells; no reading gives both.

### Claims that use the selection of responses (blocked, shown)

These claims depend on `torque-related-fractions`, which depends on `false-positive-rate`, and are
blocked with it. Their results under the claims file's detection are computed here so that a later
attempt, or the authors, can see them; they decide nothing.

In [ ]:
studied["tr"] = studied.fle_responds.fillna(False).astype(bool) | studied.ext_responds.fillna(False).astype(bool)
exp = {"M1": (75, 68), "PTN": (55, 68), "CSN": (45, 21)}
for g, m in groups.items():
    s = studied[m]
    f = s.groupby("post").tr.agg(["sum", "size", "mean"])
    print(f"{g}: pre {int(f.loc[False, 'sum'])}/{int(f.loc[False, 'size'])} ({100 * f.loc[False, 'mean']:.1f}%, expected {exp[g][0]}), "
          f"post {int(f.loc[True, 'sum'])}/{int(f.loc[True, 'size'])} ({100 * f.loc[True, 'mean']:.1f}%, expected {exp[g][1]})")
studied["tr_ttest"] = studied.fle_ttest_responds.fillna(False).astype(bool) | studied.ext_ttest_responds.fillna(False).astype(bool)
print("variant (t-test of single trials):", (100 * studied.groupby("post").tr_ttest.mean()).round(1).to_dict())

In [ ]:
def yates(table):
    chi2, p, _, _ = stats.chi2_contingency(np.asarray(table), correction=True)
    return chi2, p

for g, m in groups.items():
    s = studied[m]
    t = pd.crosstab(s.post, s.tr)
    chi2, p = yates(t)
    print(f"{g}: torque-related pre {100 * s[~s.post].tr.mean():.1f}%, post {100 * s[s.post].tr.mean():.1f}%, "
          f"Yates chi2 = {chi2:.2f}, p = {p:.3g}")

In [ ]:
s = studied[studied.ptype.isin([PTN, CSN])]
t = pd.crosstab(s.ptype, s.tr)
chi2, p = yates(t)
print(f"PTN {100 * s[s.ptype == PTN].tr.mean():.1f}% (expected 62.1), CSN {100 * s[s.ptype == CSN].tr.mean():.1f}% (expected 30.4); "
      f"Yates chi2 = {chi2:.1f}, p = {p:.2g}")

In [ ]:
tr = studied[studied.tr]
one = tr.fle_responds.fillna(False).astype(bool) ^ tr.ext_responds.fillna(False).astype(bool)
print(f"one direction only: {one.sum()}/{len(tr)} ({100 * one.mean():.1f}%, expected 44)")

In [ ]:
rows = []
for _, c in studied.iterrows():
    for d in ["fle", "ext"]:
        if c.get(f"{d}_responds") is True:
            rows.append({"file": c.file, "unit": c.unit, "post": c.post, "ptype": c.ptype, "direction": d,
                         "date": c.date, "baseline_rate": c.baseline_rate,
                         **{k: c[f"{d}_{k}"] for k in ["onset", "offset", "sign", "mean_change", "max_change",
                                                         "area", "mean_rate", "fwhm"]}})
resp = pd.DataFrame(rows)
resp["increase"] = resp.sign > 0
print(f"{len(resp)} responses in {resp.groupby(['file', 'unit']).ngroups} cells")
for g, sel in [("M1", resp.index == resp.index), ("PTN", resp.ptype == PTN)]:
    s = resp[sel]
    print(f"{g}: increases pre {100 * s[~s.post].increase.mean():.1f}%, post {100 * s[s.post].increase.mean():.1f}%")
s = resp[resp.ptype == PTN]
chi2, p = yates(pd.crosstab(s.post, s.increase))
print(f"PTN increases against decreases, pre against post: Yates chi2 = {chi2:.2f}, p = {p:.3g}")
print("expected: M1 86 and 88, PTN 70 and 75")

In [ ]:
for g, sel in [("M1", resp.index == resp.index), ("PTN", resp.ptype == PTN)]:
    s = resp[sel]
    for meas in ["mean_change", "max_change", "area"]:
        a, b = s[~s.post][meas], s[s.post][meas]
        print(f"{g} {meas}: pre median {a.median():.2f}, post {b.median():.2f}; KS p = {stats.ks_2samp(a, b).pvalue:.3g}, "
              f"Mann-Whitney p = {stats.mannwhitneyu(a, b, alternative='two-sided').pvalue:.3g}")

In [ ]:
exp = {("M1", "fle"): (42.4, 37.9), ("M1", "ext"): (50.9, 42.6), ("PTN", "fle"): (33.4, 26.1), ("PTN", "ext"): (38.8, 29.3)}
for reading in ["max_change", "mean_rate", "mean_change"]:
    out = []
    for (g, d), e in exp.items():
        s = resp[(resp.direction == d) & ((resp.ptype == PTN) if g == "PTN" else True)]
        out.append(f"{g} {d} {s[~s.post][reading].mean():.1f}/{s[s.post][reading].mean():.1f} (exp {e[0]}/{e[1]})")
    print(reading + ": " + "; ".join(out))

In [ ]:
exp = {("M1", "fle"): (39.2, 34.56), ("M1", "ext"): (40.4, 33.4), ("PTN", "fle"): (43.9, 35.5), ("PTN", "ext"): (44.4, 36.7)}
for (g, d), e in exp.items():
    s = resp[(resp.direction == d) & ((resp.ptype == PTN) if g == "PTN" else True)]
    a, b = s[~s.post].onset, s[s.post].onset
    print(f"{g} {d}: pre {a.mean():.1f} ms (exp {e[0]}, n {len(a)}), post {b.mean():.1f} (exp {e[1]}, n {len(b)}), "
          f"KS p = {stats.ks_2samp(a, b).pvalue:.3g}")
earliest = resp.groupby(["file", "unit", "post", "ptype", "date"]).onset.min().reset_index()
for g, s in [("M1", earliest), ("PTN", earliest[earliest.ptype == PTN])]:
    print(f"{g} earliest per cell: pre {s[~s.post].onset.mean():.1f}, post {s[s.post].onset.mean():.1f} ms")

In [ ]:
earliest["day"] = (pd.to_datetime(earliest.date) - first_day).dt.days
rho, p = stats.spearmanr(earliest.day, earliest.onset)
print(f"latency against days since the first Venus session: Spearman rho = {rho:.3f}, p = {p:.3g}, n = {len(earliest)}")
for post, s in earliest.groupby("post"):
    r, pp = stats.spearmanr(s.day, s.onset)
    print(f"  {'post' if post else 'pre'} only: rho = {r:.3f}, p = {pp:.3g}, n = {len(s)}")

In [ ]:
exp = {"fle": (97, 52), "ext": (57, 41)}
for d, e in exp.items():
    s = resp[(resp.ptype == PTN) & (resp.direction == d)]
    inc = s[s.increase]
    a, b = inc[~inc.post].fwhm, inc[inc.post].fwhm
    print(f"PTN {d} increases: FWHM pre {a.mean():.1f} ms (exp {e[0]}, n {len(a)}), post {b.mean():.1f} (exp {e[1]}, n {len(b)}), "
          f"Mann-Whitney p = {stats.mannwhitneyu(a, b, alternative='two-sided').pvalue:.3g}; all responses "
          f"{s[~s.post].fwhm.mean():.1f} and {s[s.post].fwhm.mean():.1f}")

In [ ]:
def dsi(c):
    f, e = c.fle_max_change_signed, c.ext_max_change_signed
    p_, np_ = (f, e) if abs(f) >= abs(e) else (e, f)
    return abs(1 - np_ / p_)

def category(c):
    fr, er = c.fle_responds is True, c.ext_responds is True
    if fr != er:
        return "unidirectional"
    if c.fle_sign != c.ext_sign:
        return "reciprocal"
    return "bidirectional" if c.dsi > 0.5 else "non-directional"

tr = studied[studied.tr].copy()
tr["dsi"] = tr.apply(dsi, axis=1)
tr["category"] = tr.apply(category, axis=1)
t = pd.crosstab(tr.post, tr.category)
print(t)
chi2, p, _, _ = stats.chi2_contingency(t)
hi = tr.category.isin(["reciprocal", "unidirectional"])
print(f"reciprocal + unidirectional: pre {100 * hi[~tr.post].mean():.1f}%, post {100 * hi[tr.post].mean():.1f}%; "
      f"chi2 = {chi2:.1f}, p = {p:.3g}")

In [ ]:
exp = {"M1": (0.72, 0.66), "PTN": (0.73, 0.75), "CSN": (0.81, 0.83)}
for g, e in exp.items():
    s = tr if g == "M1" else tr[tr.ptype == (PTN if g == "PTN" else CSN)]
    print(f"{g}: DSI pre {s[~s.post].dsi.mean():.2f} (exp {e[0]}, n {(~s.post).sum()}), "
          f"post {s[s.post].dsi.mean():.2f} (exp {e[1]}, n {s.post.sum()})")

In [ ]:
groups_dsi = [g.baseline_rate.values for _, g in tr.groupby("category")]
F, p = stats.f_oneway(*groups_dsi)
print(f"baseline rate across {tr.category.nunique()} categories: F = {F:.2f}, p = {p:.3g}")
tr["cat3"] = tr.category.replace({"reciprocal": "high", "unidirectional": "high"})
F3, p3 = stats.f_oneway(*[g.baseline_rate.values for _, g in tr.groupby("cat3")])
print(f"variant, three groups: F = {F3:.2f}, p = {p3:.3g}")

In [ ]:
vel = kin_sessions.set_index(["file", "direction"]).peak_velocity
for d, dn in [("fle", "flexion"), ("ext", "extension")]:
    s = resp[resp.direction == d]
    v = np.array([vel.get((f, dn), np.nan) for f in s.file])
    s = s[(v >= 130) & (v <= 200)]
    print(f"{dn}: responses from sessions with mean peak velocity 130 to 200 deg/s: pre {(~s.post).sum()}, post {s.post.sum()}")

In [ ]:
emg = pasq.session_emg(venus)
ptn_lat = earliest[earliest.ptype == PTN].groupby("file").onset.mean()
v_both = kin_sessions.groupby("file").peak_velocity.mean()
m2_tl1 = emg[(emg.channel == "EMGTricepsLongus1") & ~emg.has_nan & (emg.baseline_sd >= 1)].set_index("file").m2
for label, other in [("peak velocity", v_both), ("triceps longus 1 m2", m2_tl1)]:
    j = pd.concat([ptn_lat, other], axis=1, join="inner").dropna()
    rho, p = stats.spearmanr(j.iloc[:, 0], j.iloc[:, 1])
    print(f"mean PTN latency against {label}: rho = {rho:.3f}, p = {p:.3g}, n = {len(j)} sessions")

**Result: blocked (methods underspecified), through false-positive-rate.** The numbers above use a
detection whose false-positive rate is 0.23 per cell-direction, about 20 times the paper's, and
decide nothing. No session has a mean peak velocity in the 130 to 200 deg/s range of
velocity-matched-magnitude.

### kinematics-slower

> After MPTP, torque-induced displacements were slower and smaller; mean peak velocity fell by
> 16.2% and amplitude by 6.6% (ANOVA, F > 515 and F > 49).

Criterion: session means per direction, monkey V; two-way ANOVA (type II) of MPTP state and
direction with their interaction; the change is the post mean over the pre mean, minus 1;
readings (a) both directions pooled, (b) flexion only. Kinematic parameters: `pasq.py` docstring.

In [ ]:
def anova(df, meas):
    model = smf.ols(f"{meas} ~ C(post) * C(direction)", data=df).fit()
    return sm.stats.anova_lm(model, typ=2)

print("sessions:", kin_sessions.groupby(["post", "direction"]).size().to_dict())
print("perturbations without a movement onset within 200 ms:", int(kin_trials.latency.isna().sum()), "of", len(kin_trials))
for meas in ["peak_velocity", "amplitude"]:
    a = anova(kin_sessions, meas)
    pooled = 100 * (kin_sessions[kin_sessions.post][meas].mean() / kin_sessions[~kin_sessions.post][meas].mean() - 1)
    f = kin_sessions[kin_sessions.direction == "flexion"]
    flex = 100 * (f[f.post][meas].mean() / f[~f.post][meas].mean() - 1)
    print(f"\n{meas}: change (a) pooled {pooled:+.1f}%, (b) flexion {flex:+.1f}%; MPTP F = {a.loc['C(post)', 'F']:.1f}, "
          f"p = {a.loc['C(post)', 'PR(>F)']:.2g}")
    print(a.round(4))

A variant, not deciding: the pulse of the torque command is about three times larger after MPTP
(cell `finding-torque-command-levels`). Restricting to perturbations whose command pulse is 60 to
250 units, the range both states share, gives the comparison at similar torques.

In [ ]:
matched = kin_trials[kin_trials.command.between(60, 250)]
ms = matched.groupby(["file", "post", "direction"])[["latency", "peak_velocity", "amplitude"]].mean().reset_index()
print("perturbations kept:", matched.groupby("post").size().to_dict(), "; sessions:", ms.groupby("post").size().to_dict())
for meas in ["peak_velocity", "amplitude", "latency"]:
    a = anova(ms, meas)
    ch = 100 * (ms[ms.post][meas].mean() / ms[~ms.post][meas].mean() - 1)
    print(f"{meas}: pre {ms[~ms.post][meas].mean():.2f}, post {ms[ms.post][meas].mean():.2f}, change {ch:+.1f}%, "
          f"MPTP F = {a.loc['C(post)', 'F']:.1f}, p = {a.loc['C(post)', 'PR(>F)']:.2g}")

**Result: not reproduced.** Under both readings the displacements are faster and larger after
MPTP, not slower and smaller (peak velocity +28.1% pooled and +29.7% for flexion; amplitude +14.8%
and +11.1%; MPTP F = 98.5 and 25.0). Restricted to command pulses of 60 to 250 units the increase
remains (+47.0% and +40.4%), so the command's arbitrary units do not explain it.

### kinematic-latency-unchanged

> The latency of torque-induced displacements did not change after MPTP (F < 0.2, p > 0.05).

In [ ]:
a = anova(kin_sessions, "latency")
print(kin_sessions.groupby(["post", "direction"]).latency.mean().round(2))
print(a.round(4))

**Result: not reproduced.** The movement onset latency is shorter after MPTP, MPTP effect F = 709,
p < 1e-100; the MPTP by direction interaction is F = 16.1 (the paper's F > 94). At command pulses
of 60 to 250 units (variant above) the latency is also shorter after MPTP (F = 238.6).

### EMG claims

Session-average EMG of the three triceps channels for flexion torques (claims). A channel is used
in a session when no window holds a NaN and its baseline SD is at least 1 A/D unit; the claims
file does not say whether this is the SD of the session-average trace (main, the SD the response
threshold uses) or of the single-trial samples (shown as a variant).

In [ ]:
def emg_used(rule):
    return emg[~emg.has_nan & (emg[rule] >= 1)]

print("channel-sessions:", emg.groupby(["channel", "post"]).size().to_dict())
print("with NaN in a window:", int(emg.has_nan.sum()))
for rule in ["baseline_sd", "raw_sd"]:
    print(f"used under {rule} >= 1:", emg_used(rule).groupby(["channel", "post"]).size().to_dict())

In [ ]:
for rule in ["baseline_sd", "raw_sd"]:
    print("rule", rule)
    for ch, g in emg_used(rule).groupby("channel"):
        a, b = g[~g.post].m2, g[g.post].m2
        print(f"  {ch}: m2 median pre {a.median():.1f}, post {b.median():.1f} (a.u.), n {len(a)} and {len(b)}, "
              f"KS p = {stats.ks_2samp(a, b).pvalue:.2g}")

**Result: reproduced.** m2 is larger after MPTP in all three triceps channels (KS p < 1e-14 in
each) under both channel rules. The torque pulses are also about three times larger after MPTP
(cell `finding-torque-command-levels`), which this comparison cannot separate from the effect of
MPTP.

In [ ]:
for rule in ["baseline_sd", "raw_sd"]:
    u = emg_used(rule)
    d = u.dropna(subset=["duration"])
    a, b = d[~d.post].duration, d[d.post].duration
    print(f"rule {rule}: duration pre {a.mean():.1f} ms (exp 57 +/- 6, n {len(a)}), post {b.mean():.1f} (exp 79 +/- 3, n {len(b)}), "
          f"KS p = {stats.ks_2samp(a, b).pvalue:.2g}; channel-sessions without a response: "
          f"{u.duration.isna().groupby(u.post).sum().to_dict()}")
    print("  by channel:", d.groupby(["channel", "post"]).duration.mean().round(1).to_dict())

**Result: partially reproduced.** Durations are longer after MPTP (22.2 against 51.5 ms, KS
p < 1e-30), but both means are far outside the paper's 57 +/- 6 and 79 +/- 3 ms.

In [ ]:
for rule in ["baseline_sd", "raw_sd"]:
    d = emg_used(rule).dropna(subset=["duration"])
    a, b = d[~d.post].onset, d[d.post].onset
    print(f"rule {rule}: onset pre {a.mean():.1f} ms (median {a.median():.0f}), post {b.mean():.1f} (median {b.median():.0f}), "
          f"KS p = {stats.ks_2samp(a, b).pvalue:.2g}")

**Result: not reproduced.** The earliest EMG responses start earlier after MPTP (44.2 against 25.0
ms on average, KS p < 1e-20).

In [ ]:
for rule in ["baseline_sd", "raw_sd"]:
    u = emg_used(rule)
    for post, g in u.groupby("post"):
        print(f"rule {rule}, {'post' if post else 'pre'}: m2 > m1 in {(g.m2 > g.m1).sum()}/{len(g)} "
              f"({100 * (g.m2 > g.m1).mean():.1f}%), median m1/m2 {(g.m1 / g.m2).median():.2f}")

**Result: partially reproduced.** After MPTP m2 exceeds m1 in 98.3% of channel-sessions and the
median m1/m2 is 0.25; before MPTP the median ratio is 0.14 but m2 exceeds m1 in 74.0%, just under
the 75% the criterion asks (74.3% under the other channel rule).

### trial-correlations

> Trial by trial, PTN response latencies did not correlate with the peak velocity or the m2 EMG
> response (Spearman |rho| < 0.34, p > 0.05).

The Methods define a response's onset on the trial-averaged SDF only. The trials a single-trial
latency would be needed for:

In [ ]:
ptn_resp = resp[resp.ptype == PTN]
n_trials = sum(int(studied.loc[(studied.file == r.file) & (studied.unit == r.unit), f"n_{r.direction}"].iloc[0])
               for r in ptn_resp.itertuples())
print(f"PTN responses (claims file's detection): {len(ptn_resp)}, over {n_trials} perturbation trials")

**Result: blocked (methods underspecified).** The criterion needs a single-trial response latency,
which the Methods do not define; it also uses the response selection blocked above.

## Summary

| Claim | Outcome |
|---|---|
| cell-counts | reproduced |
| monkey-l-perturbations | blocked (data missing) |
| baseline-rate | reproduced |
| ptn-faster-than-csn | reproduced |
| antidromic-latency | partially reproduced |
| false-positive-rate | blocked (methods underspecified) |
| 15 claims on the response selection | blocked (methods underspecified), results shown |
| kinematics-slower | not reproduced |
| kinematic-latency-unchanged | not reproduced |
| emg-m2-larger | reproduced |
| emg-duration-longer | partially reproduced |
| emg-onset-unchanged | not reproduced |
| emg-m1-weak | partially reproduced |
| trial-correlations | blocked (methods underspecified) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: monkey V only; units are this dataset's files' units;
the response detection is the claims file's, whose false-positive rate is about 20 times the
paper's, so no claim that rests on the response selection is decided; kinematic and EMG
parameters the paper does not give are the claims file's or standard choices (`pasq.py`).

What was found about the dataset: the counts of cells by state and projection type match the
paper's monkey V table; no recording is deposited twice; the Leu files hold no perturbation
times; the torque command's pulses are about three times larger after MPTP than before, and the
evoked displacements (about 3 to 4 degrees) are smaller than the paper's 10 degrees and faster,
not slower, after MPTP.